# Decorators

Welcome! Decorators wrap functions or classes to add behavior. This guide covers function decorators, class decorators, decorators with arguments, the descriptor protocol, and common patterns.

## Functions as Objects

Functions are first-class objects. They can be passed around and returned.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

f = greet
print(f("Alice"))

def apply(func, arg):
    return func(arg)

print(apply(greet, "Bob"))

## Basic Function Decorators

A decorator is a function that takes a function and returns a wrapped function.

In [ ]:
def log(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Returned {result}")
        return result
    return wrapper

@log
def add(a, b):
    return a + b

add(2, 3)

# Same as: add = log(add)

## functools.wraps

Use `@wraps(func)` to preserve the original function's metadata.

In [ ]:
from functools import wraps

def log(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log
def add(a, b):
    """Add two numbers."""
    return a + b

print(add.__name__, add.__doc__)

## Decorators with Arguments

Use a factory: a function that returns a decorator.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hi():
    print("Hi!")

say_hi()

## Class Decorators

Decorators can wrap classes too. They receive the class and return a modified class.

In [ ]:
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Config:
    def __init__(self):
        self.value = 42

c1 = Config()
c2 = Config()
print(c1 is c2)

## Descriptor Protocol

Descriptors define `__get__`, `__set__`, `__delete__`. Used by `property`, `staticmethod`, `classmethod`.

In [ ]:
class ValidatedString:
    def __init__(self, min_len=0):
        self.min_len = min_len

    def __set_name__(self, owner, name):
        self.name = f"_{name}"

    def __get__(self, obj, objtype=None):
        return getattr(obj, self.name, "")

    def __set__(self, obj, value):
        if len(value) < self.min_len:
            raise ValueError(f"Min length {self.min_len}")
        setattr(obj, self.name, value)

class Person:
    name = ValidatedString(min_len=2)

p = Person()
p.name = "Alice"
print(p.name)
# p.name = "A"  # ValueError

## Common Decorator Patterns

Caching, timing, retry, validation.

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_func():
    time.sleep(0.1)
    return "done"

slow_func()

## Stacking Decorators

Decorators are applied bottom-to-top. The innermost runs first when called.

In [ ]:
def bold(func):
    def wrapper():
        return "<b>" + func() + "</b>"
    return wrapper

def italic(func):
    def wrapper():
        return "<i>" + func() + "</i>"
    return wrapper

@bold
@italic
def hello():
    return "Hello"

print(hello())

## Built-in Decorators

`@property`, `@staticmethod`, `@classmethod`, `@functools.lru_cache`, `@dataclasses.dataclass`.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=32)
def expensive(n):
    return n ** 2

print(expensive(5))
print(expensive(5))
print(expensive.cache_info())

## Summary

You've learned function decorators, class decorators, decorators with args, the descriptor protocol, common patterns, and built-in decorators. Decorators enable clean, reusable cross-cutting concerns.